# 02 — Cohort and Retention Proxy Analysis

Retention analysis using activity and recency proxies.

**Project:** Marketing Analytics Causal & LTV Lab  
**Phase:** Phase 1 — Customer Analytics, Retention, Churn and LTV Baseline

> This notebook is designed as a hands-on learning notebook. Run each section, inspect the output, and discuss the interpretation before moving to the next step.


## 1. Notebook objective

This notebook performs retention-style analysis using a customer-level dataset.

Important limitation: the dataset does not contain signup dates or transaction-level timestamps. Therefore, we cannot create true calendar cohorts. Instead, we create **proxy cohorts** based on activity, recency, usage and customer attributes.


In [ ]:
# Core imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_RAW = Path("../data/raw/digital_wallet_ltv_dataset.csv")
DATA_PROCESSED = Path("../data/processed")
REPORTS = Path("../reports")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED / 'wallet_clean.csv') if (DATA_PROCESSED / 'wallet_clean.csv').exists() else pd.read_csv(DATA_RAW)
df.head()


## 2. Define retention and activity proxies


In [ ]:
df["retained_30d"] = (df["Last_Transaction_Days_Ago"] <= 30).astype(int)
df["dormant_90d"] = (df["Last_Transaction_Days_Ago"] > 90).astype(int)

df["activity_cohort"] = pd.qcut(
    df["Active_Days"].rank(method="first"),
    q=4,
    labels=["Low activity", "Medium-low activity", "Medium-high activity", "High activity"]
)

df["transaction_cohort"] = pd.qcut(
    df["Total_Transactions"].rank(method="first"),
    q=4,
    labels=["Low transactions", "Medium-low transactions", "Medium-high transactions", "High transactions"]
)

display(df[["Customer_ID", "Active_Days", "Last_Transaction_Days_Ago", "retained_30d", "dormant_90d", "activity_cohort"]].head())


## 3. Retention by activity cohort


In [ ]:
retention_by_activity = (
    df.groupby("activity_cohort")
    .agg(
        customers=("Customer_ID", "count"),
        retained_30d_rate=("retained_30d", "mean"),
        dormant_90d_rate=("dormant_90d", "mean"),
        avg_ltv=("LTV", "mean"),
        avg_total_spent=("Total_Spent", "mean")
    )
    .reset_index()
)

display(retention_by_activity)

plt.figure(figsize=(8, 4))
plt.bar(retention_by_activity["activity_cohort"].astype(str), retention_by_activity["retained_30d_rate"])
plt.title("30-Day Retention Proxy by Activity Cohort")
plt.ylabel("Retention rate")
plt.xticks(rotation=30)
plt.show()


## 4. Retention by income, location and payment method


In [ ]:
def segment_summary(df, segment_col):
    return (
        df.groupby(segment_col)
        .agg(
            customers=("Customer_ID", "count"),
            retained_30d_rate=("retained_30d", "mean"),
            dormant_90d_rate=("dormant_90d", "mean"),
            avg_ltv=("LTV", "mean"),
            avg_satisfaction=("Customer_Satisfaction_Score", "mean")
        )
        .sort_values("avg_ltv", ascending=False)
    )

for col in ["Income_Level", "Preferred_Payment_Method", "Location", "App_Usage_Frequency"]:
    if col in df.columns:
        print(f"\n=== {col} ===")
        display(segment_summary(df, col))


## 5. Retention risk matrix


In [ ]:
df["ltv_quartile"] = pd.qcut(df["LTV"].rank(method="first"), q=4, labels=["Low LTV", "Mid-low LTV", "Mid-high LTV", "High LTV"])
df["recency_bucket"] = pd.cut(
    df["Last_Transaction_Days_Ago"],
    bins=[-1, 7, 30, 90, np.inf],
    labels=["0-7 days", "8-30 days", "31-90 days", "90+ days"]
)

risk_matrix = pd.crosstab(
    df["ltv_quartile"],
    df["recency_bucket"],
    values=df["Customer_ID"],
    aggfunc="count",
    normalize="index"
)

display(risk_matrix)

plt.figure(figsize=(8, 4))
plt.imshow(risk_matrix, aspect="auto")
plt.xticks(range(len(risk_matrix.columns)), risk_matrix.columns, rotation=30)
plt.yticks(range(len(risk_matrix.index)), risk_matrix.index)
plt.title("Recency Distribution within LTV Segments")
plt.colorbar(label="Share within LTV segment")
plt.show()


## 6. Save retention features and write summary


In [ ]:
output = DATA_PROCESSED / "wallet_retention_features.csv"
df.to_csv(output, index=False)

summary = f'''
# Phase 1 Retention Summary

## Dataset limitation
This dataset is customer-level and does not contain signup dates or transaction-level timestamps.
Therefore, true calendar cohort retention cannot be estimated.

## Proxy retention definitions
- retained_30d = Last_Transaction_Days_Ago <= 30
- dormant_90d = Last_Transaction_Days_Ago > 90

## Main outputs
- Activity-based retention comparison
- Segment-level retention comparison
- LTV-recency risk matrix

## Next step
Use these proxies to create a churn target and model churn risk.
'''

(REPORTS / "phase_1_retention_summary.md").write_text(summary)
print(f"Saved: {output}")
print("Saved report: reports/phase_1_retention_summary.md")
